# 04 — TensorFlow and Keras

This notebook introduces TensorFlow tensors, automatic differentiation, and a complete Keras classification workflow. It generates synthetic data locally, so no dataset download is required.

## 1. Import and environment check

Install a TensorFlow version supported by your Python and operating system using the [official installation guide](https://www.tensorflow.org/install).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Keras API available:', hasattr(keras, 'Sequential'))

## 2. Tensor basics

A tensor is a multidimensional array. Its shape gives the size of each dimension, and its rank is the number of dimensions.

In [ ]:
vector = tf.constant([2, 4, 6, 8])
matrix = tf.constant([[1., 2.], [3., 4.]])
print('vector:', vector)
print('vector shape:', vector.shape, 'rank:', tf.rank(vector).numpy())
print('matrix shape:', matrix.shape, 'dtype:', matrix.dtype)
print('vector * 3:', (vector * 3).numpy())

## 3. Automatic differentiation with GradientTape

`tf.GradientTape` records operations and calculates derivatives using automatic differentiation.

In [ ]:
x = tf.Variable(4.0)
with tf.GradientTape() as tape:
    f = 3 * x**2 + 2 * x + 1
gradient = tape.gradient(f, x)
print('f(4) =', f.numpy())
print('df/dx at x=4 =', gradient.numpy())  # 6*4 + 2 = 26

## 4. Generate a synthetic binary-classification dataset

We create a reproducible dataset with 8 numeric features. The label is based on a nonlinear combination of features plus noise. This is for learning the API, not a benchmark.

In [ ]:
rng = np.random.default_rng(SEED)
n_samples, n_features = 1200, 8
X_all = rng.normal(size=(n_samples, n_features)).astype('float32')
logits = (1.4 * X_all[:, 0] - 1.1 * X_all[:, 1] + 0.8 * X_all[:, 2] * X_all[:, 3] + 0.3 * rng.normal(size=n_samples))
y_all = (logits > np.median(logits)).astype('float32')

# Shuffle and split before any data-dependent preprocessing.
indices = rng.permutation(n_samples)
train_end, val_end = int(0.70*n_samples), int(0.85*n_samples)
train_idx, val_idx, test_idx = indices[:train_end], indices[train_end:val_end], indices[val_end:]
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]
X_test, y_test = X_all[test_idx], y_all[test_idx]
print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

## 5. Define and compile a Keras model

For binary classification, a single sigmoid output produces a probability between 0 and 1. Binary cross-entropy is a common loss for this setup.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 6. Train with callbacks

Early stopping monitors validation loss and restores the best weights. The validation set is used during model development; the test set is held back for final evaluation.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=6, restore_best_weights=True
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
print('Epochs run:', len(history.history['loss']))
print('Final recorded training loss:', history.history['loss'][-1])
print('Final recorded validation loss:', history.history['val_loss'][-1])

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.title('Training and validation loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Evaluate on held-out test data

Do not use test metrics to repeatedly tune the model. Use validation metrics for model-development decisions.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
probabilities = model.predict(X_test[:10], verbose=0).ravel()
predicted_classes = (probabilities >= 0.5).astype(int)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print('First 10 probabilities:', np.round(probabilities, 3))
print('First 10 classes:      ', predicted_classes)
print('First 10 true labels:  ', y_test[:10].astype(int))

## 8. Save and reload the model

The native `.keras` format is recommended for saving a Keras model. This cell writes a local file named `binary_classifier.keras` in the notebook's working directory.

In [ ]:
model_path = 'binary_classifier.keras'
model.save(model_path)
reloaded_model = keras.models.load_model(model_path)
original_predictions = model.predict(X_test[:20], verbose=0)
reloaded_predictions = reloaded_model.predict(X_test[:20], verbose=0)
max_difference = np.max(np.abs(original_predictions - reloaded_predictions))
print('Saved to:', model_path)
print('Maximum prediction difference after reload:', float(max_difference))
assert np.allclose(original_predictions, reloaded_predictions, atol=1e-6)

## 9. Optional experiment

Try changing the hidden units, learning rate, or batch size. Re-run training and compare validation loss. Change one factor at a time and record your observations. Avoid using test performance to choose hyperparameters.

## Key takeaways

- TensorFlow provides tensors and automatic differentiation.
- Keras offers a high-level API for model definition, compilation, training, evaluation, and saving.
- Match the output layer and loss function to the target format.
- Use validation data during development and reserve test data for final evaluation.
- Save and reload models to verify a portable inference workflow.

Next: complete `practice.md`, then check `solution.md`.